In [ ]:
from datetime import datetime, timedelta
import numpy as np
import pycountry
import pycountry_convert as pc
import json
from IPython.display import display, Markdown
from matplotlib import pyplot as plt
from matplotlib.pyplot import cm

from emu_renewal.run import find_variant_seeds, find_run_start_time, find_run_end_time
from emu_renewal.inputs import DATA_PATH, TEXT_DATE_FORMAT, \
    get_worldbank_national_pop, get_country_vacc_data, \
    get_country_vars, find_relevant_vars, get_google_mobility, get_fb_mobility, \
    get_apple_mobility, get_prealpha_prop, \
    get_indicator_series_from_who_data, get_filtered_seroprev, get_country_hosps
from emu_renewal.utils import get_row_proportions

In [ ]:
all_countries = json.load(open(DATA_PATH / f"config/included.json", "r"))
country = "Austria"
start_plot_time = datetime(2020, 1, 1)
end_plot_time = datetime(2021, 12, 31)
iso2 = pycountry.countries.lookup(country).alpha_2
continent = pc.country_alpha2_to_continent_code(iso2)

In [ ]:
# Gather common data
iso3 = pycountry.countries.lookup(country).alpha_3
country_name = pycountry.countries.lookup(country).name
pop = get_worldbank_national_pop(iso3, 2020)
deaths_start_threshold = 2e-6
min_var_threshold = 10
cases_target = get_indicator_series_from_who_data("New_cases", country)
deaths_target = get_indicator_series_from_who_data("New_deaths", country)
data_start = find_run_start_time(deaths_target, pop, deaths_start_threshold)
hosp_target, hosp_out_name = get_country_hosps(country, data_start, end_plot_time)
seroprev_target = get_filtered_seroprev(country, data_start, end_plot_time)
prealpha_prop = get_prealpha_prop(iso3, min_var_threshold)

# Create figure
fig, axes = plt.subplots(7, 1, figsize=[12, 15], sharex=True)
fig.suptitle(country_name, fontsize=18, y=1.0)

# Plot cases
case_ax = axes[0]
case_ax.plot(cases_target.index, cases_target, linewidth=0, marker=".")
case_ax.set_title("cases")

# Plot hospitalisations
hosp_ax = axes[1]
hosp_ax.plot(hosp_target.index, hosp_target, linewidth=0, marker=".", color="red")
hosp_ax.set_title("hospitalisations")

# Plot deaths
death_start_threshold = 2e-6
per_capita_deaths = deaths_target / pop
death_ax = axes[2]
death_ax.plot(deaths_target.index, deaths_target, linewidth=0, marker=".")
start_time = per_capita_deaths.index[per_capita_deaths.gt(death_start_threshold)].min()
death_ax.axvline(start_time)
death_ax.set_title("deaths")

# Plot seroprevalence
prev_ax = axes[3]
prev_ax.plot(seroprev_target.index, seroprev_target, linewidth=0, marker=".")
prev_ax.set_title("seroprevalence")
prev_ax.set_ylim([0.0, 1.0])

# Plot mobility
g_mob = get_google_mobility(iso3)
a_mob = get_apple_mobility(iso3)
f_mob = get_fb_mobility(iso3)
mob_ax = axes[4]
mob_ax.plot(g_mob.index, g_mob, label="google", color="blue")
mob_ax.plot(a_mob.index, a_mob, label="apple", color="red")
mob_ax.plot(f_mob.index, f_mob, label="facebook", color="green")
mob_ax.set_title("mobility")

# Plot variants
min_var_samples = 2
if iso3 == "CZE":
    country = pycountry.countries.lookup(iso3).official_name
elif iso3 == "USA":
    country = pycountry.countries.lookup(iso3).alpha_3
else:
    country = pycountry.countries.lookup(iso3).name
var_data = get_country_vars(country)
var_data = var_data[var_data.sum(axis=1) >= min_var_samples]
var_props = get_row_proportions(var_data)
var_ax = axes[5]
var_ax.stackplot(var_props.index, var_props.T, colors=cm.turbo(np.linspace(0.0, 1.0, len(var_props.columns))), labels=var_props.columns)
var_ax.legend(ncols=2)
eu_prop = get_prealpha_prop(iso3, min_var_samples)
var_ax.plot(eu_prop.index, eu_prop, linewidth=0, marker=".", markerfacecolor="w", markersize=10)
run_start = find_run_start_time(deaths_target, pop, 2e-6)
min_var_samples = 10
seed_times = find_variant_seeds(0.5, eu_prop, run_start)
var_ax.axvline(seed_times[0])
var_ax.legend().remove()
var_ax.set_title("variants")

# Plot vaccination
vacc_data = get_country_vacc_data(iso3)
cov_threshold = 0.05
vacc_ax = axes[6]
vacc_ax.plot(vacc_data.index, vacc_data)
cov_time = find_run_end_time(vacc_data, cov_threshold, continent)
vacc_ax.axvline(cov_time)
vacc_ax.set_ylim(0.0, 100.0)
vacc_ax.set_title("vaccination")

# Tidy figure
case_ax.set_xlim([start_plot_time, end_plot_time])
plt.setp(axes[-1].xaxis.get_majorticklabels(), rotation=70)
fig.tight_layout()

# Report text results
display(Markdown(f"### {country_name}\nISO3: {iso3}\n"))
display(Markdown(f"Population: {round(pop / 1e6, 3)} million\n"))
display(Markdown(f"Date at which {death_start_threshold * 1e6} deaths per million per day exceeded: {start_time.strftime(TEXT_DATE_FORMAT)}"))
# display(Markdown(f"Date at which {round(cov_threshold * 100)}% vaccination coverage exceeded: {cov_time.strftime(TEXT_DATE_FORMAT)}"))
display(Markdown(f"Hospitalisation indicator: {hosp_out_name}"))